# Co-Scientist: Method Overview & Results

This notebook documents every training method, its insight, pipeline, and results.

**Task:** Train an LLM (Qwen3-30B-A3B) to generate structured research plans given a research goal. A grader model scores outputs using a 7-desiderata rubric per criterion (levels 0–3).

**Dataset:** `facebook/research-plan-gen` (ml split). Fields: `Goal`, `Rubric` (list of rubric items), `Reference solution`.

**Shared reward formula (all methods):**
```
reward = rubric_score + 0.08 * length_bonus - format_penalty

rubric_score  = mean over items of (mean of 7 mapped desiderata levels)
level_map     = {0: 0.0, 1: 0.2, 2: 0.6, 3: 1.0}
length_bonus  = exp(-((word_count - 600) / 120)^2)     # Gaussian, peak at 600 words
format_penalty = 0.0 if compliant else 0.2 + 0.0005 * excess_words
```

**Baseline:** Reference solutions score **0.87 mean rubric** (graded with simple grader). Base model (no training) scores **0.56**.

---
## Phase 1: Paper Reproduction (arXiv:2512.xxxxx)

### Insight
Reproduce the original co-scientist paper's RL training method as a baseline.

### Pipeline
- Standard GRPO training on research plan generation
- Strict reproduction of the paper's methodology

### Result
- **Did NOT improve** task success rate in this setting
- Conclusion: the paper's method alone is insufficient — motivates custom improvements (Phase 2)

---
## Phase 2: Bestversion (`best_ver.py`)

### Insight
Two custom modifications (a1 + a2) on top of the base GRPO pipeline yield a stronger single-stage trainer. The key idea is a well-tuned reward signal combining rubric scoring with a Gaussian length bonus and format penalties, trained with single-stage GRPO over 8 samples per goal.

### Pipeline

```
┌─────────────────────────────────────────────────────────┐
│  For each batch of 64 goals:                            │
│                                                         │
│  Phase 1: Generate 8 plans per goal (async)             │
│     prompt = scenario + guidelines + <think>/<solution> │
│                                                         │
│  Phase 2: Grade each plan (simple grader, greedy)       │
│     grader sees: scenario, rubric items, reference sol  │
│     outputs XML: 7 desiderata × N items → levels 0-3   │
│                                                         │
│  Phase 3: Compute rewards & GRPO advantages             │
│     reward = rubric + 0.08*length_bonus - format_pen    │
│     advantage = reward_i - mean(group_rewards)          │
│     filter degenerate samples (<30 words, bare tags)    │
│                                                         │
│  Phase 4: PPO-clipped optimization step                 │
│     clip_eps = 0.2, lr = 1e-5, LoRA rank 64            │
└─────────────────────────────────────────────────────────┘
```

### Config
| Parameter | Value |
|-----------|-------|
| Model | Qwen/Qwen3-30B-A3B |
| batch_size | 64 |
| group_size | 8 |
| learning_rate | 1e-5 |
| clip_eps | 0.2 |
| lora_rank | 64 |
| max_tokens (policy) | 2048 |
| grader_max_tokens | 8192 |
| temperature | 1.0 (policy) / 0.0 (grader) |

### Key files
- Training: `src/co_scientist/trainers/baselines/best_ver.py`
- Eval: `src/co_scientist/trainers/baselines/eval_only.py`
- Runs: `runs/2026/2/withA1,A2/2(ml)/`

### Results

**Training trajectory (215 batches = 2 epochs):**

| Batch | Train Rubric | Train Reward |
|-------|-------------|-------------|
| 0 | 0.6141 | 0.6519 |
| 107 (mid) | 0.7324 | 0.7949 |
| 214 (end) | 0.7270 | 0.7835 |

- Reward peaks ~0.83 around batch 139, then mild oscillation
- Format compliance: ~99%

**Eval:**
| Checkpoint | Eval Rubric (simple grader) |
|------------|---------------------------|
| Base model (no training) | 0.56 |
| Batch 214 (2 epochs) | **0.69** |
| Reference solutions | 0.87 |

**Status:** Performance ceiling reached at 0.69. Further extensions of this single-stage approach failed to surpass it. Gap to reference: **0.18**.

---
## Phase 3: Double Generation (`train_double_generation.py`)

### Insight
Break the bestversion ceiling by adding a refinement stage. Instead of generating once, the model generates an initial draft (Stage 1), receives structured feedback from a detailed grader, then revises (Stage 2). Both stages are trained jointly via GRPO with independent advantage groups. Compute-matched: 4 initial + 4 refined = 8 total samples per goal (same compute budget as bestversion's 8 single-stage samples).

### Pipeline

```
┌──────────────────────────────────────────────────────────────────┐
│  For each batch of 64 goals:                                     │
│                                                                  │
│  Phase 1: Generate 4 initial plans per goal (async)              │
│                                                                  │
│  Phase 2: Grade each with DETAILED grader (async)                │
│     detailed grader adds: <sample_review>, <weaknesses>,         │
│     <actionable_fixes>, <desiderata_review> per item             │
│                                                                  │
│  Phase 3: Extract feedback from grader XML                       │
│     → weaknesses + fixes (filtered for specificity)              │
│     → only items with desiderata level ≤ 2                       │
│                                                                  │
│  Phase 4: Build refinement prompt per plan                       │
│     prompt = scenario + rubric items + initial draft (solution   │
│     only, think stripped) + expert critique + revision rules     │
│     Stage 2 output: <solution> only (no <think> required)        │
│     Launch 4 refinement generations per goal (async)              │
│                                                                  │
│  Phase 5: Grade refined plans with detailed grader               │
│                                                                  │
│  Phase 6: Compute GRPO advantages (independently per stage)      │
│     Group A (initial):  adv = (reward - mean) / std              │
│     Group B (refined):  adv = (reward - mean) / std              │
│     Combined into single PPO-clipped optimizer step              │
└──────────────────────────────────────────────────────────────────┘
```

### Key design differences from bestversion
| Aspect | Bestversion | Double Generation |
|--------|------------|-------------------|
| Stages | 1 | 2 (initial → refine) |
| Samples/goal | 8 | 4+4 (compute-matched) |
| Grader | Simple | Detailed (for feedback) |
| Advantage normalization | `reward - mean` | `(reward - mean) / std` |
| Async model | `concurrent.futures` | Full `asyncio` |
| Stage 2 output | N/A | `<solution>` only (no `<think>`) |
| Stage 2 sees | N/A | Rubric items + feedback + draft |

### Key files
- Training: `src/co_scientist/trainers/refinement/train_double_generation.py` (2061 lines)
- Eval: `src/co_scientist/trainers/baselines/eval_only_double.py`
- Runs: `runs/2026/3/refinement/8/` (epoch 2)
- Analysis: `analysis/notebooks/double_gen_diagnosis.ipynb`

### Results

**Training trajectory (epoch 2, 179 batches):**

| Batch | Initial Rubric | Refined Rubric | Delta | Improvement Rate | Regression Rate | Format |
|-------|---------------|---------------|-------|-----------------|-----------------|--------|
| 0 | 0.2355 | 0.4964 | 0.2609 | 89.5% | 10.2% | 75.0% |
| 44 | 0.2676 | 0.5474 | 0.2799 | 88.3% | 10.2% | 88.7% |
| 89 | 0.3450 | 0.5573 | 0.2124 | 80.9% | 16.8% | 94.5% |
| 130 | 0.3877 | 0.6176 | 0.2299 | 84.8% | 14.8% | 91.6% |
| 139 (peak) | 0.4133 | 0.6187 | 0.2054 | 82.4% | 16.8% | 91.4% |
| 160 | 0.3878 | 0.5599 | 0.1721 | 71.9% | 27.3% | 86.7% |
| 178 (end) | 0.3369 | 0.4701 | 0.1333 | 78.1% | 21.5% | 85.7% |

**NOTE:** Training rubric scores use the **detailed grader**, which produces lower scores than the simple grader. These numbers are NOT directly comparable to bestversion's training rubric.

**Eval (simple grader, comparable to bestversion):**
| Checkpoint | Eval Rubric |
|------------|------------|
| After epoch 1 | **0.65** (still rising) |

### Degradation analysis (epoch 2)

The model collapses after batch ~139. Root cause:

1. **Stage 1 improves** throughout training (initial rubric: 0.24 → 0.41)
2. This causes **distribution shift** in Stage 2 input — Stage 2 now receives better drafts than it was trained on
3. Stage 2's learned refinement strategy becomes **out-of-distribution** → advantage signal collapses → noisy gradients
4. **Regression rate doubles**: 10% → 28% — Stage 2 actively hurts samples
5. **Refinement delta shrinks**: 0.26 → 0.13 — refinement adds less value
6. **Feedback bullets decrease**: 21.7 → 20.8 — grader gives less actionable feedback on better plans

**Status:** Promising early results (0.65 eval after epoch 1, refined rubric peaks 0.62) but unstable. The two-stage coupling creates a distribution shift problem that causes collapse in extended training.

---
## Phase 4: Rubric Dropout (`best_ver_rubric_dropout.py`) — CURRENT

### Insight

The remaining gap between the model (0.69) and reference solutions (0.87) is largely due to **information asymmetry**: reference solutions are written by experts who know the evaluation criteria, but the model never sees the rubric items during generation. It must blindly guess what the grader will check.

**Rubric dropout** closes this gap through curriculum-style training:
- With probability `p`, the model sees the rubric criteria in its prompt (rubric-visible) → learns what good criterion coverage looks like
- With probability `1-p`, the rubric is hidden (rubric-hidden) → the model must infer criteria autonomously from the scenario
- `p` is **signal-driven**: it tracks what fraction of the initial reward gap remains (`p = gap / reference_gap`), so it decays naturally as the model learns to infer criteria — no manual decay rates or thresholds
- At eval time, rubric is NEVER shown — matching the real task

### Pipeline

```
┌───────────────────────────────────────────────────────────────────┐
│  Same 4-phase GRPO loop as bestversion, with one key difference: │
│                                                                   │
│  Prompt construction per goal:                                    │
│    coin flip with probability p (from RubricScheduler)            │
│                                                                   │
│    IF visible (p):                                                │
│      prompt += "# Evaluation Criteria" + rubric items             │
│      <think> → "Review each criterion and reason about how       │
│               your plan addresses it"                             │
│                                                                   │
│    IF hidden (1-p):                                               │
│      <think> → "Identify key technical requirements,              │
│               constraints, and criteria an evaluator would check" │
│                                                                   │
│  After each batch:                                                │
│    Split rewards by visible/hidden                                │
│    Update RubricScheduler → new p                                 │
│    Log rubric/p, gap, reference_gap, visible/hidden counts        │
│                                                                   │
│  RubricScheduler (signal-driven):                                 │
│    gap = EMA(reward_visible) - EMA(reward_hidden)                 │
│    reference_gap = gap measured at training start                 │
│                                                                   │
│    Lifecycle:                                                     │
│      1. Bootstrap (batches 0-1): p = 0.5, collect both EMAs      │
│      2. First measurement: record reference_gap, keep p = 0.5    │
│      3. Signal-driven: p = clamp(gap / reference_gap, p_min, 1)  │
│                                                                   │
│    As training closes the gap → p drops proportionally            │
│    If gap > reference_gap (model regressed) → p clamped to 1.0   │
│    If gap ≤ 0 (hidden beats visible) → p = p_min                 │
└───────────────────────────────────────────────────────────────────┘
```

### Config
| Parameter | Default | Purpose |
|-----------|---------|---------|
| `rubric_prob_min` | 0.05 | Floor for p (>0 prevents dead state where visible EMA starves) |

All other config params (batch_size, group_size, lr, etc.) inherited from bestversion.

### Key files
- Training: `src/co_scientist/trainers/baselines/best_ver_rubric_dropout.py`
- Eval: `src/co_scientist/trainers/baselines/eval_only.py` (same eval as bestversion — ensures comparable scores)
- Runs: `runs/2026/3/rubric_dropout/12/` (planned)

### Results
Training not yet started.

---
## Gap Analysis: Selection vs Capability

Analysis of bestversion eval (batch 214, 552 goals × 8 samples):

```
Model mean:      0.70
Model best-of-8: 0.85  ← already near reference
Reference:       0.87

Selection gap (mean → best-of-8): 0.157  (91% of total gap)
Capability gap (best-of-8 → ref): 0.015  (9% of total gap)
```

**91% of the gap is the model not picking its best plan, not lacking the ability to produce one.**

Best-of-N ceiling for varying N:
| N | Mean best-of-N rubric |
|---|----------------------|
| 1 | 0.70 |
| 2 | 0.77 |
| 4 | 0.82 |
| 8 | 0.85 |

Per-item failure analysis (items scoring < 1.0):
- 80.6% of model-low items: reference scores HIGH (>= 2.5) on the same item
- 1.1% of model-low items: reference also scores low (inherently hard)
- Root cause: rubric items check for **method-specific** details from the reference solution; the model proposes different valid approaches

---

## Best-of-N with Self-Selection (`eval_bon.py`)

### Insight
The model already PRODUCES 0.85-level plans (best-of-8) but averages only 0.69 because it doesn't know which of its 8 samples is best. A self-evaluator that infers likely criteria and selects the best plan could close this selection gap — without any additional training.

### Pipeline
```
┌───────────────────────────────────────────────────────────────┐
│  For each batch of 64 goals:                                  │
│                                                               │
│  Phase 1: Generate 8 plans per goal (same as eval_only.py)    │
│                                                               │
│  Phase 2: Launch in parallel for each plan:                   │
│    a) Grader (with rubric + reference) → rubric score         │
│    b) Self-eval (scenario + plan only, NO rubric) → score     │
│                                                               │
│  Self-evaluator prompt:                                       │
│    1. Infer what criteria an expert would check                │
│    2. Score plan against inferred criteria using same          │
│       7 desiderata as the real grader                         │
│    3. Output XML in same format → parsed by same function     │
│                                                               │
│  Phase 3: For each goal, select plan with highest self-eval   │
│                                                               │
│  Phase 4: Report three metrics:                               │
│    mean-of-N:    avg rubric across all samples (= current)    │
│    oracle BoN:   best sample by grader score (upper bound)    │
│    self-selected: sample chosen by self-eval score            │
│    + rank correlation, gap closed %, oracle match rate         │
└───────────────────────────────────────────────────────────────┘
```

### Config
| Parameter | Default | Purpose |
|-----------|---------|---------|
| `self_eval_model` | "base" | "base" (grader model) or "finetuned" (policy checkpoint) |
| `self_eval_max_tokens` | 8192 | Token budget for self-eval reasoning |
| `self_eval_temperature` | 0.0 | Deterministic scoring |

All other params inherited from eval_only.py.

### Key files
- Eval: `src/co_scientist/trainers/baselines/eval_bon.py`
- Imports from: `eval_only.py` (grader prompt, reward, checkpoint)

### Results
Not yet run.

---

## Summary Comparison

| | Phase 1: Reproduction | Phase 2: Bestversion | Phase 3: Double Gen | Phase 4: Rubric Dropout | BoN Self-Select |
|---|---|---|---|---|---|
| **Core idea** | Reproduce paper | a1+a2 improvements | Draft → feedback → refine | Criteria-aware curriculum | Infer-then-select |
| **Type** | Training | Training | Training | Training | Eval-time |
| **Stages** | 1 | 1 | 2 | 1 | 1 + selection |
| **Samples/goal** | — | 8 | 4+4 | 8 | 8 (select 1) |
| **Grader** | — | Simple | Detailed | Simple | Simple + self-eval |
| **Eval rubric** | no improvement | **0.69** | 0.65 (epoch 1) | TBD | TBD |
| **Ceiling** | — | 0.69 | — | ~0.71-0.72 | 0.85 (oracle) |
| **Key limitation** | Didn't work | Ceiling at 0.69 | Distribution shift | Small capability gap | Self-eval quality |

**Reference solutions:** 0.87 &nbsp;&nbsp;|&nbsp;&nbsp; **Base model (no training):** 0.56

---

## Eval Scripts Reference

| Script | Purpose | Grader used | Notes |
|--------|---------|-------------|-------|
| `eval_only.py` | Single-pass eval | Simple | Canonical eval for all methods |
| `eval_only_double.py` | Two-stage eval (Stage 1 → feedback → Stage 2) | Simple (scoring) + Detailed (feedback) | Scores both stages with simple grader |
| `eval_reference.py` | Grade reference solutions | Simple | Result: 0.87 mean rubric |
| `eval_bon.py` | Best-of-N with self-selection | Simple (scoring) + self-eval (no rubric) | Reports mean, oracle, self-selected |

All eval scripts are in `src/co_scientist/trainers/baselines/`. Always use `eval_only.py` for single-stage methods (bestversion, rubric dropout) to ensure score comparability.